# VolleyVision: fine-tune WASB (ball finder) on your checked clips

[WASB](https://github.com/nttcom/WASB-SBDT) (BMVC 2023, MIT) is a small HRNet that finds the ball from 3 frames.
Out of the box (trained on broadcast volleyball) it is ~2.5x more precise than our TrackNet and reports the fewest
false balls, but finds only 54–71% of your balls. This notebook teaches it your gyms.

Uses the **same export as the TrackNet notebook** (old TrackNet set + your checked clips):

    cd training/tracknet/datasets && tar -xf tracknet_v1.tar && cd ../../..
    python -m tools.export_training --name v2 --only tracknet --tracknet-base training/tracknet/datasets/tracknet_data \
        --tracknet-size 640x288 --valid test49 test54 test59 --archive

**Put in Google Drive under `MyDrive/VolleyVision/`:**
- `tracknet_v2.tar` (from `training/tracknet/datasets/`)
- `wasb_volleyball_best.pth.tar` (from `models/wasb/`)
- a folder `wasb_code/` with 3 files from your project: `core/wasb.py`, `core/wasb_hrnet.py`, `training/wasb/train_wasb.py`

Runtime → Change runtime type → **GPU** (A100 / L4 if offered; T4 works with a smaller batch).

In [ ]:
# ---- settings ----
DATA = 'tracknet_v2'                     # the tar's name without .tar
INIT = 'wasb_volleyball_best.pth.tar'    # start from the authors' volleyball weights (in MyDrive/VolleyVision)
WIDTH, HEIGHT = 640, 288                 # your clips' shape at WASB's 288 px height (multiples of 8)
EPOCHS = 15
BATCH = 8                                # A100/L4: 8 (or 16); T4: 4. "CUDA out of memory" -> halve it
OUT = 'wasb_ft_v2'                       # checkpoints go to MyDrive/VolleyVision/<OUT>/
# fewer false balls (heads, hands, clocks): 1 / 0 = off
NEG_WEIGHT = 1                           # e.g. 3: windows with no visible ball are drawn 3x as often
MINE_FROM = 0                            # e.g. 4: from epoch 4 on, mine the windows where it fires on a non-ball
HARD_WEIGHT = 3                          # ... and draw those 3x as often
MINE_EVERY = 1                           # re-mine every N epochs (mining costs ~half an epoch)
LR = 1e-4                                # use 3e-5 when continuing from an already fine-tuned checkpoint
ZOOM_PROB = 0.3                          # share of windows zoomed in 1.5-3x: teaches WASB big, near-camera balls
# round 3 (NEG_WEIGHT 3, HARD_WEIGHT 3, mining every epoch) was too strong: recall fell 0.87 -> 0.77.
# gentler retry (continue from round 2):  INIT = 'wasb_ft2.pth.tar'; EPOCHS = 3; LR = 3e-5; NEG_WEIGHT = 1.5; MINE_FROM = 2; MINE_EVERY = 2; HARD_WEIGHT = 2; OUT = 'wasb_ft_v4'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/VolleyVision'
import os
for f in [f'{DATA}.tar', INIT, 'wasb_code/wasb.py', 'wasb_code/wasb_hrnet.py', 'wasb_code/train_wasb.py']:
    print(('OK      ' if os.path.exists(f'{D}/{f}') else 'MISSING ') + f)

In [ ]:
# code -> /content/vv (same layout as the project), data -> /content/data
!mkdir -p /content/vv/core /content/vv/training/wasb /content/data
!cp {D}/wasb_code/wasb.py {D}/wasb_code/wasb_hrnet.py /content/vv/core/
!cp {D}/wasb_code/train_wasb.py /content/vv/training/wasb/
!tar -xf {D}/{DATA}.tar -C /content/data
!ls /content/data/{DATA}; head -c 400 /content/data/{DATA}/manifest.json

## Train
Prints validation F1 / precision / recall **before** fine-tuning (the authors' weights on your validation clips),
then after every epoch. `best.pth.tar` = best validation F1; `last.pth.tar` is saved every epoch for resuming.

In [ ]:
%cd /content/vv
!python training/wasb/train_wasb.py --data /content/data/{DATA} --init {D}/{INIT} --out {D}/{OUT} \
    --epochs {EPOCHS} --batch {BATCH} --width {WIDTH} --height {HEIGHT} --workers 2 --lr {LR} --neg-weight {NEG_WEIGHT} --mine-from {MINE_FROM} --mine-every {MINE_EVERY} --hard-weight {HARD_WEIGHT} --zoom-prob {ZOOM_PROB}

## Done
Copy `MyDrive/VolleyVision/wasb_ft_v2/best.pth.tar` to `models/wasb/wasb_ft2.pth.tar` on the Mac (it carries its
input size, so the pipeline runs it at 640x288 automatically), then compare on your labelled clips:

    python -m tools.compare_finders --runs --finders tracknet wasb wasb:models/wasb/wasb_ft2.pth.tar

## If the runtime disconnected
Re-run the setup cells (everything above **Train**), then this cell: it continues from `last.pth.tar`.

In [ ]:
%cd /content/vv
!python training/wasb/train_wasb.py --data /content/data/{DATA} --out {D}/{OUT} \
    --epochs {EPOCHS} --batch {BATCH} --width {WIDTH} --height {HEIGHT} --workers 2 --lr {LR} --neg-weight {NEG_WEIGHT} --mine-from {MINE_FROM} --mine-every {MINE_EVERY} --hard-weight {HARD_WEIGHT} --zoom-prob {ZOOM_PROB} --resume